# İSPARK Otopark Verisiyle Basit Doğrusal Regresyon (Sıfırdan)

Bu defterde İstanbul'daki İSPARK otoparklarının verisini kullanarak **basit doğrusal regresyonu**
formülünden başlayarak öğreneceğiz. Amaç: bir otopark noktasının "gerçek otopark alanı" mı
yoksa "yol üstü/taksi/minibüs parkı" mı olduğunun, park **kapasitesini** ne kadar açıkladığını bulmak.

## Regresyonun temel mantığı

Basit doğrusal regresyon, bir bağımlı değişkeni (y) bir bağımsız değişkenden (x) şu denklemle tahmin eder:

$$\hat{y} = b_0 + b_1 x$$

- **b0 (intercept/sabit):** x = 0 iken y'nin tahmini değeri
- **b1 (eğim/slope):** x bir birim arttığında y'nin ortalama ne kadar değiştiği

Bu doğruyu bulmak için **En Küçük Kareler (Least Squares)** yöntemini kullanırız — gerçek gözlemler
ile tahminler arasındaki farkların karelerinin toplamını minimize eden doğruyu buluruz:

$$b_1 = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2} \qquad b_0 = \bar{y} - b_1 \bar{x}$$


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.max_columns', None)


## 1. Veriyi yükleme ve keşif

Önce veriyi yüklüyor, boyutuna ve eksik değerlere bakıyoruz.

In [ ]:
df = pd.read_csv('ispark_parking.csv')
print(df.shape)
df.head()

In [ ]:
print(df.isnull().sum())

`isnull()` hiç eksik değer göstermiyor. Ama gerçek veriyle çalışırken bu yeterli değil —
koordinatların gerçek İstanbul sınırları içinde olup olmadığını kontrol edelim.

In [ ]:
bad_coords = df[(df['LATITUDE'] < 30) | (df['LONGITUDE'] < 20)]
bad_coords[['PARK_NAME', 'COUNTY_NAME', 'LONGITUDE', 'LATITUDE', 'CAPACITY_OF_PARK']]

4 satırda `LONGITUDE` ve `LATITUDE` değeri **-99**. Bu, "eksik veri" için kullanılan klasik bir
**sentinel (yer tutucu) değer** — NaN değil ama aslında geçersiz. Regresyondan önce bu tip gizli
hataları temizlemek şart, çünkü modelinizi tamamen çarpıtabilir.

In [ ]:
df = df[(df['LATITUDE'] > 30) & (df['LONGITUDE'] > 20)].copy()
print('Temizlenmiş satır sayısı:', len(df))

Şimdi park tipine göre kapasite ortalamalarına bakalım.

In [ ]:
df.groupby('PARK_TYPE_DESC')['CAPACITY_OF_PARK'].agg(['count', 'mean', 'median', 'std'])

Fark çok belirgin: gerçek bir "otopark alanı" olan yerler (açık/kapalı otopark), yol kenarındaki
park yerlerinden ortalama 9-10 kat daha yüksek kapasiteli. Bu, regresyon için mantıklı bir hipotez veriyor.

## 2. Hipotez ve değişken hazırlama

**Hipotez:** Bir park noktasının "gerçek otopark alanı" olup olmaması, kapasitesini anlamlı şekilde
etkiler mi?

`PARK_TYPE_DESC` kategorik (metin) bir değişken; regresyon sayısal girdi ister. Bu yüzden bir
**dummy (gösterge) değişken** oluşturuyoruz — kategorik veriyi regresyona sokmanın standart yolu budur.

In [ ]:
df['IS_LOT'] = df['PARK_TYPE_DESC'].isin(
    ['AÇIK OTOPARK', 'KAPALI OTOPARK']
).astype(int)
# 1 = otopark alanı, 0 = yol üstü / taksi / minibüs parkı

df[['PARK_TYPE_DESC', 'IS_LOT']].drop_duplicates()

## 3. Regresyonu sıfırdan hesaplama

Şimdi yukarıdaki en küçük kareler formüllerini elle uyguluyoruz — hiçbir regresyon kütüphanesi
kullanmadan.

In [ ]:
x = df['IS_LOT'].values.astype(float)
y = df['CAPACITY_OF_PARK'].values.astype(float)

b1 = np.sum((x - x.mean()) * (y - y.mean())) / np.sum((x - x.mean())**2)
b0 = y.mean() - b1 * x.mean()

print(f"b0 (intercept) = {b0:.2f}")
print(f"b1 (slope)     = {b1:.2f}")
print(f"Denklem: ŷ = {b0:.1f} + {b1:.1f}·x")

**Yorumu:**
- **b0 ≈ 43.5** → yol üstü/taksi/minibüs parklarının ortalama kapasitesi ~43.5 araç
- **b1 ≈ 355.4** → bir yer "otopark alanı" ise, kapasite ortalama 355 araç daha fazla
  (43.5 + 355.4 ≈ 399, gruplama tablosundaki ortalamayla birebir örtüşüyor ✓)

x sadece 0 veya 1 alabildiği için bu regresyon aslında iki grubun ortalamasını karşılaştırıyor —
basit regresyonun en temel hali budur.

## 4. Model ne kadar iyi? (R²)

In [ ]:
y_pred = b0 + b1 * x
ss_res = np.sum((y - y_pred)**2)    # açıklanamayan varyans
ss_tot = np.sum((y - y.mean())**2)  # toplam varyans
r2 = 1 - ss_res / ss_tot

print(f"R² = {r2:.4f}")

**R² ≈ 0.13** demek: park tipi, kapasitedeki farklılığın yalnızca **%13'ünü** açıklıyor.
Geri kalan %87 başka faktörlere bağlı (konum, ilçe, alanın fiziksel büyüklüğü gibi bizim veri
setimizde olmayan değişkenler).

Bu önemli bir ders: istatistiksel olarak anlamlı olmak ile modelin güçlü bir tahminci olması aynı
şey değildir. Bunu bir sonraki hücrede p-değeriyle de doğrulayalım.

## 5. Doğrulama: scipy ile karşılaştırma

In [ ]:
sonuc = stats.linregress(x, y)
print(f"slope (b1)   = {sonuc.slope:.4f}")
print(f"intercept(b0)= {sonuc.intercept:.4f}")
print(f"R²           = {sonuc.rvalue**2:.4f}")
print(f"p-değeri     = {sonuc.pvalue:.3e}")

Elle hesapladığımız b0, b1 ve R² değerleri scipy'nin sonucuyla birebir aynı ✓.

p-değeri (≈ 2.9×10⁻²³) son derece küçük — yani "otopark alanı olup olmaması" ile kapasite
arasındaki ilişki kesinlikle tesadüf değil, istatistiksel olarak anlamlı. Ama gördüğümüz gibi
R² düşük: anlamlı olması, tek başına iyi bir tahminci olduğu anlamına gelmiyor.

## 6. Görselleştirme

In [ ]:
np.random.seed(42)
jitter = np.random.uniform(-0.08, 0.08, size=len(x))
resid = y - y_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.scatter(x + jitter, y, alpha=0.4, s=20, color='#3b6ea5', label='Gerçek gözlemler')
xs = np.array([0, 1])
ax.plot(xs, b0 + b1 * xs, color='#d64545', linewidth=2.5,
        label=f'Regresyon doğrusu: ŷ = {b0:.1f} + {b1:.1f}·x')
ax.set_xticks([0, 1])
ax.set_xticklabels(['Yol Üstü / Taksi / Minibüs (0)', 'Otopark Alanı (1)'])
ax.set_ylabel('Kapasite (araç sayısı)')
ax.set_title('Basit Doğrusal Regresyon: Otopark Tipi → Kapasite')
ax.set_ylim(-100, 2000)  # birkaç aşırı uç değeri sınırlıyoruz (görsel amaçlı)
ax.legend(fontsize=8)

ax2 = axes[1]
ax2.scatter(y_pred + np.random.uniform(-2, 2, len(x)), resid, alpha=0.4, s=20, color='#3b6ea5')
ax2.axhline(0, color='#d64545', linewidth=2)
ax2.set_xlabel('Tahmin edilen değer (ŷ)')
ax2.set_ylabel('Artık (residual) = y - ŷ')
ax2.set_title('Artık (Residual) Grafiği')

plt.tight_layout()
plt.show()

**Soldaki grafik:** İki grup arasındaki net kapasite farkını ve regresyon doğrusunu gösteriyor
(birkaç aşırı büyük otopark — örneğin 5000 kapasiteli biri — görüş netliği için grafik sınırının
dışında bırakıldı, ama hesaplamalara dahil).

**Sağdaki grafik (residual/artık):** Gerçek değer ile tahmin arasındaki farkları gösteriyor.
Noktaların 0 çizgisinin hem üstünde hem altında geniş bir dağılım göstermesi, modelin açıklayamadığı
büyük varyansın kanıtı — R²'nin düşük çıkmasının görsel karşılığı bu.

## Özet — sıfırdan öğrendiklerin

1. Regresyon = en küçük kareler ile en iyi uyan doğruyu bulma
2. Kategorik değişkenler dummy (0/1) kodlanarak regresyona sokulur
3. b0/b1 katsayıları somut, yorumlanabilir anlamlar taşır
4. R² model gücünü, p-değeri ilişkinin tesadüf olup olmadığını gösterir — ikisi farklı şeyler
5. Gerçek veride tek değişken genelde yeterli değildir → bir sonraki adım **çoklu regresyon**
   (ilçe, konum gibi değişkenler eklemek)
